# Manual Implementation of Support Vector Machine (Linear)

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('train.csv')
data.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [3]:
features = ['pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine','winddirection', 'windspeed']
target = 'rainfall'

In [4]:
# Fill missing values for features
for feature in features:
    data[feature] = data[feature].fillna(data[feature].mean())

In [5]:
X_raw = data[features].values
X_raw

array([[1.0174e+03, 2.1200e+01, 2.0600e+01, ..., 1.1000e+00, 6.0000e+01,
        1.7200e+01],
       [1.0195e+03, 1.6200e+01, 1.6900e+01, ..., 0.0000e+00, 5.0000e+01,
        2.1900e+01],
       [1.0241e+03, 1.9400e+01, 1.6100e+01, ..., 8.3000e+00, 7.0000e+01,
        1.8100e+01],
       ...,
       [1.0133e+03, 1.9000e+01, 1.6300e+01, ..., 5.0000e+00, 4.0000e+01,
        3.2900e+01],
       [1.0223e+03, 1.6400e+01, 1.5200e+01, ..., 1.0000e-01, 4.0000e+01,
        1.8000e+01],
       [1.0138e+03, 2.1200e+01, 1.9100e+01, ..., 1.0000e+00, 7.0000e+01,
        4.8000e+01]], shape=(2190, 10))

In [6]:
Y = data[target].values
Y

array([1, 1, 1, ..., 1, 1, 1], shape=(2190,))

In [7]:
X_scaled = np.zeros_like(X_raw)

X_means = np.mean(X_raw, axis=0)
X_stds = np.std(X_raw, axis=0)
X_stds[X_stds == 0] = 1e-8

for j in range(len(features)):
    X_scaled[:, j] = (X_raw[:, j] - X_means[j]) / X_stds[j]

print(X_scaled)

[[ 0.67170214 -0.91380916 -0.64219876 ... -0.72939738 -0.56090052
  -0.4652908 ]
 [ 1.04311572 -1.79828913 -1.35084574 ... -1.03280391 -0.6859253
   0.00962944]
 [ 1.85668833 -1.23222195 -1.50406671 ...  1.25653632 -0.43587575
  -0.37434863]
 ...
 [-0.05343865 -1.30298035 -1.46576147 ...  0.34631671 -0.81095007
   1.12114491]
 [ 1.53833383 -1.76290993 -1.6764403  ... -1.0052215  -0.81095007
  -0.38445331]
 [ 0.03499315 -0.91380916 -0.92948808 ... -0.75697979 -0.43587575
   2.6469525 ]]


In [8]:
Y_svm = np.where(Y == 0, -1, 1)
Y_svm

array([1, 1, 1, ..., 1, 1, 1], shape=(2190,))

## Prediction / Hypothesis

In [9]:
def predict_distance(x_row, w, b):
    z = b + np.dot(w, x_row)
    return z

## Cost Function

$cost\_function = \frac{1}{m} \sum_{i=1}^{n} \max(0, 1 - y_i(h_wx)) + \frac{\lambda}{2} \|w\|^2$

$h_wx = (w^T x_i + b)$

In [10]:
def cost_function(X, Y, w, b, lambda_reg):
    m = len(Y)
    current_cost = 0.0

    for i in range(m):
        h_wx = predict_distance(X[i], w, b)
        
        # The Hinge Loss formula: max(0, 1 - y*z)
        margin = 1 - (Y[i] * h_wx)
        if margin > 0:
            current_cost += margin
            
    # Total Cost = Average Hinge Loss + L2 Penalty
    l2_penalty = (lambda_reg / 2) * sum([w_j**2 for w_j in w])
    return (current_cost / m) + l2_penalty

## Weight Update

In [11]:
def update_weights(X, Y, w, b, learning_rate, lambda_reg):
    m = len(Y)
    n = len(w)

    w_deriv = [0.0] * n
    b_deriv = 0.0
    
    for i in range(m):
        h_wx = predict_distance(X[i], w, b)
        
        if Y[i] * h_wx < 1: #Violates margin, penalize
            b_deriv += -Y[i]
            for j in range(n):
                w_deriv[j] += -Y[i] * X[i][j]
                
        # If Y[i] * z >= 1, the margin is satisfied, and the Hinge gradient is 0.
        # So we do nothing to the derivatives for this data point!

    # Update Bias
    b -= learning_rate * (b_deriv / m)
    
    # Update Weights (L2 Regularization)
    for j in range(n):
        total_deriv = (w_deriv[j] / m) + (lambda_reg * w[j])
        w[j] -= learning_rate * total_deriv

    return w, b

## Training loop

In [12]:
m = len(Y_svm)
n = len(X_scaled[0])

# Initialization
w = [0.0] * n 
b = 0.0

learning_rate = 0.01
epochs = 5000
tolerance = 0.0001
lambda_reg = 0.01

print("Starting Support Vector Machine Training...\n")
prev_cost = cost_function(X_scaled, Y_svm, w, b, lambda_reg)

for epoch in range(epochs):
    w, b = update_weights(X_scaled, Y_svm, w, b, learning_rate, lambda_reg)
    current_cost = cost_function(X_scaled, Y_svm, w, b, lambda_reg)

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}: SVM Cost= {current_cost:.5f}")

    # Convergence Check
    cost_decrease = prev_cost - current_cost
    
    # Because Hinge Loss has sharp mathematical corners, the cost can sometimes slightly bounce up.
    # We ensure the decrease is positive but smaller than our tolerance to break.
    if 0 < cost_decrease < tolerance:
        print(f"\n[Convergence Reached] Stopped early at Epoch {epoch}!")
        break

    prev_cost = current_cost

print("\nTraining Complete!")
print("-" * 50)
print(f"Final Bias (b) : {b:.4f}")
for j in range(n):
    print(f"Weight {features[j]:15} : {w[j]:.4f}")

Starting Support Vector Machine Training...

Epoch    0: SVM Cost= 0.99032
Epoch    1: SVM Cost= 0.98064
Epoch   50: SVM Cost= 0.54454
Epoch  100: SVM Cost= 0.40566
Epoch  150: SVM Cost= 0.36637
Epoch  200: SVM Cost= 0.35078
Epoch  250: SVM Cost= 0.34202

[Convergence Reached] Stopped early at Epoch 288!

Training Complete!
--------------------------------------------------
Final Bias (b) : 0.7196
Weight pressure        : -0.0434
Weight maxtemp         : 0.0091
Weight temparature     : 0.0168
Weight mintemp         : 0.0267
Weight dewpoint        : 0.1061
Weight humidity        : 0.1653
Weight cloud           : 0.5579
Weight sunshine        : -0.3706
Weight winddirection   : 0.0313
Weight windspeed       : 0.0274


In [13]:
# To predict for a specific scaled day:
test_day = X_scaled[4]

h_wx = predict_distance(test_day, w, b)

if h_wx >= 0:
    print(f"Distance: {h_wx:.2f} -> Prediction: RAIN (1)")
else:
    print(f"Distance: {h_wx:.2f} -> Prediction: NO RAIN (-1)")

Distance: -1.21 -> Prediction: NO RAIN (-1)


## Test Data

In [14]:
test_data = pd.read_csv('test.csv') 

# NaN value removal with mean
for feature in features:
    test_data[feature] = test_data[feature].fillna(test_data[feature].mean())

In [15]:
X_test_raw = test_data[features].values
X_test_raw

array([[1.0195e+03, 1.7500e+01, 1.5800e+01, ..., 0.0000e+00, 5.0000e+01,
        2.4300e+01],
       [1.0165e+03, 1.7500e+01, 1.6500e+01, ..., 0.0000e+00, 5.0000e+01,
        3.5300e+01],
       [1.0239e+03, 1.1200e+01, 1.0400e+01, ..., 0.0000e+00, 4.0000e+01,
        1.6900e+01],
       ...,
       [1.0227e+03, 2.1000e+01, 1.8500e+01, ..., 0.0000e+00, 5.0000e+01,
        2.1900e+01],
       [1.0144e+03, 2.1000e+01, 2.0000e+01, ..., 0.0000e+00, 5.0000e+01,
        3.9500e+01],
       [1.0209e+03, 2.2200e+01, 1.8800e+01, ..., 2.0000e-01, 6.0000e+01,
        5.0600e+01]], shape=(730, 10))

In [16]:
# feature scaling in testing data
X_test_scaled = np.zeros_like(X_test_raw)
for j in range(len(features)):
    X_test_scaled[:, j] = (X_test_raw[:, j] - X_means[j]) / X_stds[j]

X_test_scaled

array([[ 1.04311572, -1.56832434, -1.56152458, ..., -1.03280391,
        -0.6859253 ,  0.25214191],
       [ 0.5125249 , -1.56832434, -1.42745623, ..., -1.03280391,
        -0.6859253 ,  1.36365737],
       [ 1.8213156 , -2.6827691 , -2.59576612, ..., -1.03280391,
        -0.81095007, -0.49560486],
       ...,
       [ 1.60907927, -0.94918836, -1.0444038 , ..., -1.03280391,
        -0.6859253 ,  0.00962944],
       [ 0.14111132, -0.94918836, -0.75711449, ..., -1.03280391,
        -0.6859253 ,  1.78805418],
       [ 1.29072478, -0.73691317, -0.98694594, ..., -0.97763909,
        -0.56090052,  2.90967433]], shape=(730, 10))

# Scikit-learn built-in SVM

In [17]:
from sklearn.svm import SVC

In [18]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)

print("Training Scikit-Learn SVM...")
svm_model.fit(X_scaled, Y_svm)

Training Scikit-Learn SVM...


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [19]:

b_builtin_svm = svm_model.intercept_[0]
w_builtin_svm = svm_model.coef_[0]

print("\n--- Model Parameters ---")
print(f"Sklearn Bias    : {b_builtin_svm:.4f}")
print("Weights:")
for j in range(len(features)):
    print(f"Weight {features[j]:15}: {w_builtin_svm[j]:.4f}")


--- Model Parameters ---
Sklearn Bias    : 0.8502
Weights:
Weight pressure       : -0.0557
Weight maxtemp        : 0.1314
Weight temparature    : -0.0044
Weight mintemp        : -0.1649
Weight dewpoint       : 0.2718
Weight humidity       : 0.0838
Weight cloud          : 0.9805
Weight sunshine       : -0.1295
Weight winddirection  : -0.0138
Weight windspeed      : 0.0434


In [20]:
# Manual Model Prediction
manual_svm_predictions = []
for i in range(len(X_test_scaled)):
    h_wx = predict_distance(X_test_scaled[i], w, b)
    manual_svm_predictions.append(1 if h_wx >= 0 else -1)

manual_svm_predictions = np.array(manual_svm_predictions)

In [21]:
# Predictions from Scikit-Learn SVM
sklearn_svm_predictions = svm_model.predict(X_test_scaled)

In [22]:



# Calculate the Agreement Rate
agreements = np.sum(manual_svm_predictions == sklearn_svm_predictions)
total_samples = len(X_test_scaled)
agreement_rate = (agreements / total_samples) * 100

print("\n--- Head-to-Head SVM Agreement ---")
print(f"Total Test Samples : {total_samples}")
print(f"Models Agreed On   : {agreements}")
print(f"Agreement Rate     : {agreement_rate:.2f}%")


--- Head-to-Head SVM Agreement ---
Total Test Samples : 730
Models Agreed On   : 712
Agreement Rate     : 97.53%


In [23]:
# Visual Comparison
comparison_df = pd.DataFrame({
    'id': test_data['id'],
    'Manual_SVM': manual_svm_predictions,
    'Sklearn_SVM': sklearn_svm_predictions,
    'Match': manual_svm_predictions == sklearn_svm_predictions
})

print("\n--- First 15 Predictions Head-to-Head ---")
print(comparison_df.head(15))


--- First 15 Predictions Head-to-Head ---
      id  Manual_SVM  Sklearn_SVM  Match
0   2190           1            1   True
1   2191           1            1   True
2   2192           1            1   True
3   2193          -1           -1   True
4   2194          -1           -1   True
5   2195           1            1   True
6   2196           1            1   True
7   2197           1            1   True
8   2198           1            1   True
9   2199           1            1   True
10  2200           1            1   True
11  2201          -1           -1   True
12  2202           1            1   True
13  2203           1            1   True
14  2204          -1           -1   True
